In [2]:

# CONNECT PROJECT ROOT

import sys
from pathlib import Path

# Detect project root using README.md as marker
current = Path.cwd()
root = None

while current != current.parent:
    if (current / "README.md").exists():
        root = current
        break
    current = current.parent

if root is None:
    raise RuntimeError("Project root not found. Ensure README.md exists.")

# Add root to Python path
if str(root) not in sys.path:
    sys.path.insert(0, str(root))


# IMPORT PROJECT UTILITIES

from src.utils.config import get_path, ensure_path

import pandas as pd
import re


# DEFINE ANONYMIZATION FUNCTION

# Remove personally identifiable information (PII)
def anonymize_text(text):
   
    if not isinstance(text, str) or pd.isnull(text):
        return text

    # Email addresses
    text = re.sub(r'\S+@\S+', '[EMAIL]', text)

    # Kenyan phone numbers
    text = re.sub(r'\b(?:\+254|0)?7\d{8}\b', '[PHONE]', text)

    # 8-digit ID numbers
    text = re.sub(r'\b\d{8}\b', '[ID_NUMBER]', text)

    # Long account numbers
    text = re.sub(r'\b\d{10,}\b', '[ACCOUNT_NUMBER]', text)

    # Simple name detection (capitalized words)
    text = re.sub(r'\b[A-Z][a-z]+\b', '[NAME]', text)

    return text


# LOAD DATA

raw_folder = get_path("data", "raw_data")
output_folder = ensure_path("data", "anonymized_data")

excel_files = list(raw_folder.glob("*.xlsx"))

if not excel_files:
    raise FileNotFoundError(f"No Excel files found in {raw_folder}")



# COMBINE FILES

df_list = []

for file_path in excel_files:
    try:
        df = pd.read_excel(file_path, engine='openpyxl')
        df_list.append(df)
        print(f"Loaded: {file_path.name}")
    except Exception as e:
        print(f"Skipped {file_path.name}: {e}")

combined_df = pd.concat(df_list, ignore_index=True)



# APPLY ANONYMIZATION

if "text" in combined_df.columns:
    combined_df["text"] = combined_df["text"].apply(anonymize_text)
else:
    raise ValueError("Column 'text' not found in dataset.")



# SAVE OUTPUT

output_file = output_folder / "customer_feedback_anonymized.xlsx"
combined_df.to_excel(output_file, index=False, engine='openpyxl')

print(f"\nDone! Anonymized dataset saved to:\n{output_file}")


Loaded: app_store_reviews.xlsx
Loaded: Emails 2025.xlsx
Loaded: google_play_reviews.xlsx
Loaded: SMSes_2025.xlsx
Loaded: Survey Responses.xlsx

Done! Anonymized dataset saved to:
C:\Users\LENOVO\OneDrive\Documents\Strath\Masters\sentiment-analysis-tool\data\anonymized_data\customer_feedback_anonymized.xlsx
